# Race Strategy

Replay laps 30 to 50 of the 2025 Australian Grand Prix as if they were live. Watch race events stream into BigQuery, see continuous queries flag the moments that matter, and read the strategy call that Gemini makes for each one.

This notebook follows the same steps as `docs/01-race-strategy.md`. Run the cells in order. Each query here is also a saved query under **BigQuery Studio > Queries**.

In [ ]:
import time

import google.auth
from google.auth.transport.requests import AuthorizedSession
from google.cloud import bigquery

# The region and BigQuery location from infra/variables.tf (var.region and
# var.dataset_location). Change them here if you changed them there.
REGION = "us-central1"
BQ_LOCATION = "US"

credentials, PROJECT_ID = google.auth.default(
    scopes=["https://www.googleapis.com/auth/cloud-platform"]
)
session = AuthorizedSession(credentials)
bq = bigquery.Client(project=PROJECT_ID, location=BQ_LOCATION)


def run_sql(sql):
    """Run a query from this notebook, with the project filled in, and return a DataFrame."""
    return bq.query(sql.replace("${project_id}", PROJECT_ID)).to_dataframe()


print("Project:", PROJECT_ID)

## Before you start

The replays so far, one row per `run_id`. Saved query **00 Check: replays so far**.

In [ ]:
REPLAYS = r"""
WITH events AS (
  SELECT run_id, COUNT(*) AS events, MIN(publish_time) AS started, MAX(publish_time) AS last_event
  FROM `${project_id}.raw.raw_events`
  GROUP BY run_id
),

moments AS (
  SELECT run_id, COUNT(DISTINCT TO_JSON_STRING(STRUCT(moment, driver_number, event_ts))) AS moments
  FROM `${project_id}.serving.moments`
  GROUP BY run_id
),

calls AS (
  SELECT run_id, COUNTIF(status = 'OK') AS calls_ok, COUNTIF(status = 'FAILED') AS calls_failed
  FROM `${project_id}.serving.calls`
  GROUP BY run_id
)

SELECT
  run_id,
  started,
  last_event,
  events,
  IFNULL(moments, 0) AS moments,
  IFNULL(calls_ok, 0) AS calls_ok,
  IFNULL(calls_failed, 0) AS calls_failed
FROM events
LEFT JOIN moments USING (run_id)
LEFT JOIN calls USING (run_id)
ORDER BY started DESC;
"""
run_sql(REPLAYS)

## Step 1: Start the race

Run the `replayer` Cloud Run job with a new `run_id`. The job reads the race from the bucket and publishes it to the `race-events` topic at five times real time. It needs between 20 seconds and 2.5 minutes to start.

In [ ]:
RUN_ID = f"run-{int(time.time())}"
response = session.post(
    f"https://run.googleapis.com/v2/projects/{PROJECT_ID}/locations/{REGION}/jobs/replayer:run",
    json={"overrides": {"containerOverrides": [{"env": [{"name": "RUN_ID", "value": RUN_ID}]}]}},
)
response.raise_for_status()
print("Started replay", RUN_ID)

A continuous query started on an idle table fails with "Invalid timestamp bounds", so wait for the first events before you start the queries.

In [ ]:
COUNT_EVENTS = "SELECT COUNT(*) AS n FROM `${project_id}.raw.raw_events` WHERE run_id = @run_id"
config = bigquery.QueryJobConfig(
    query_parameters=[bigquery.ScalarQueryParameter("run_id", "STRING", RUN_ID)]
)
while True:
    n = next(
        iter(
            bq.query(COUNT_EVENTS.replace("${project_id}", PROJECT_ID), job_config=config).result()
        )
    )["n"]
    print(f"{n} events")
    if n > 0:
        break
    time.sleep(10)

Start the two continuous queries, as the `cq-runner` service account. These are saved queries **03a** and **03b**, which Step 3 explains.

In [ ]:
MOMENTS_FAST = r"""
EXPORT DATA OPTIONS (
  format = 'CLOUD_PUBSUB',
  uri = 'https://pubsub.googleapis.com/projects/${project_id}/topics/race-moments'
) AS (
  SELECT TO_JSON_STRING(STRUCT(run_id, moment, driver_number, lap, event_ts, detail)) AS message
  FROM (
    SELECT
      run_id,
      CASE
        WHEN event_type = 'race_control'
          AND JSON_VALUE(payload, '$.category') = 'SafetyCar'
          AND JSON_VALUE(payload, '$.message') LIKE '%DEPLOYED%' THEN 'SAFETY_CAR'
        WHEN event_type = 'pit' THEN 'PIT_STOP'
        -- The replayer adds previous_rainfall to each weather reading.
        WHEN event_type = 'weather'
          AND SAFE.INT64(payload.rainfall) = 1
          AND SAFE.INT64(payload.previous_rainfall) = 0 THEN 'RAIN_STARTED'
      END AS moment,
      driver_number,
      SAFE.INT64(payload.lap_number) AS lap,
      event_ts,
      CASE event_type
        WHEN 'race_control' THEN JSON_VALUE(payload, '$.message')
        WHEN 'pit' THEN FORMAT('pit lane %s s', JSON_VALUE(payload, '$.lane_duration'))
        WHEN 'weather' THEN 'rainfall 0 -> 1'
      END AS detail
    -- Change history lags the writes, so the start time sits 10 minutes back.
    -- A start time after the readable end fails with "Invalid timestamp bounds"
    -- (verified on 2026-10-25). A moment from an earlier run can be flagged
    -- again, so the strategist answers each moment once.
    FROM APPENDS(TABLE `${project_id}.raw.raw_events`, CURRENT_TIMESTAMP() - INTERVAL 10 MINUTE)
    WHERE event_type IN ('race_control', 'pit', 'weather')
  )
  WHERE moment IS NOT NULL
);
"""

In [ ]:
# 30 and 1 are the defaults of window_seconds and gap_closing_seconds in infra/variables.tf.
MOMENTS_WINDOWED = r"""
EXPORT DATA OPTIONS (
  format = 'CLOUD_PUBSUB',
  uri = 'https://pubsub.googleapis.com/projects/${project_id}/topics/race-moments'
) AS (
  WITH ev AS (
    SELECT
      _CHANGE_TIMESTAMP AS ct,
      run_id,
      event_ts,
      driver_number,
      SAFE.FLOAT64(payload.interval) AS gap
    -- Change history lags the writes, so the start time sits 10 minutes back.
    -- A start time after the readable end fails with "Invalid timestamp bounds".
    FROM APPENDS(TABLE `${project_id}.raw.raw_events`, CURRENT_TIMESTAMP() - INTERVAL 10 MINUTE)
    WHERE event_type = 'intervals'
  ),

  windows AS (
    SELECT
      run_id,
      driver_number,
      MIN_BY(gap, event_ts) AS gap_first,
      MAX_BY(gap, event_ts) AS gap_last,
      MAX(event_ts) AS event_ts
    FROM TUMBLE(TABLE ev, "ct", INTERVAL 30 SECOND)
    GROUP BY window_end, run_id, driver_number
  )

  SELECT TO_JSON_STRING(STRUCT(
    run_id,
    'GAP_CLOSING' AS moment,
    driver_number,
    CAST(NULL AS INT64) AS lap,
    event_ts,
    FORMAT('gap to car ahead %.3f s -> %.3f s', gap_first, gap_last) AS detail
  )) AS message
  FROM windows
  -- The gap shrinks by at least gap_closing_seconds and ends inside DRS range.
  WHERE gap_first - gap_last >= 1 AND gap_last < 1.0
);
"""

In [ ]:
JOB_IDS = []
for name, sql in [("moments-fast", MOMENTS_FAST), ("moments-windowed", MOMENTS_WINDOWED)]:
    response = session.post(
        f"https://bigquery.googleapis.com/bigquery/v2/projects/{PROJECT_ID}/jobs",
        json={
            "jobReference": {"projectId": PROJECT_ID, "location": BQ_LOCATION},
            "configuration": {
                "labels": {"pitwall": name},
                "query": {
                    "query": sql.replace("${project_id}", PROJECT_ID),
                    "useLegacySql": False,
                    "continuous": True,
                    "connectionProperties": [
                        {
                            "key": "service_account",
                            "value": f"cq-runner@{PROJECT_ID}.iam.gserviceaccount.com",
                        }
                    ],
                },
            },
        },
    )
    response.raise_for_status()
    JOB_IDS.append(response.json()["jobReference"]["jobId"])
print("Continuous queries:", JOB_IDS)

## Step 2: Look at the events landing in BigQuery

Every message on `race-events` lands in `raw.raw_events` through a BigQuery subscription. Run this cell a few times. Saved query **02 Events: what landed for the latest replay**.

In [ ]:
EVENTS = r"""
DECLARE target_run STRING DEFAULT (
  SELECT run_id FROM `${project_id}.raw.raw_events`
  WHERE event_type = 'drivers'
  ORDER BY publish_time DESC LIMIT 1);

SELECT
  event_type,
  COUNT(*) AS events,
  MAX(event_ts) AS latest_race_time,
  MAX(publish_time) AS latest_publish_time
FROM `${project_id}.raw.raw_events`
WHERE run_id = target_run
GROUP BY event_type
ORDER BY events DESC;
"""
run_sql(EVENTS)

## Step 3: Read the two continuous queries

`MOMENTS_FAST` (above) flags a safety car, a pit stop or the start of rain from one event, within seconds. `MOMENTS_WINDOWED` flags a car closing on the car ahead over a 30-second `TUMBLE` window, within about a minute.

The moments of the latest replay, with the seconds from the event landing to the moment reaching Pub/Sub. Saved query **03c Moments: what the continuous queries flagged**.

In [ ]:
MOMENTS = r"""
DECLARE target_run STRING DEFAULT (
  SELECT run_id FROM `${project_id}.raw.raw_events`
  WHERE event_type = 'drivers'
  ORDER BY publish_time DESC LIMIT 1);

SELECT
  m.moment,
  m.lap,
  d.code AS driver,
  m.event_ts AS race_time,
  m.detail,
  ROUND(TIMESTAMP_DIFF(m.publish_time, MIN(r.publish_time), MILLISECOND) / 1000, 1) AS seconds_after_event
FROM `${project_id}.serving.moments` AS m
LEFT JOIN `${project_id}.raw.raw_events` AS r
  ON r.run_id = m.run_id AND r.event_ts = m.event_ts
  AND r.driver_number IS NOT DISTINCT FROM m.driver_number
LEFT JOIN (
  SELECT driver_number, ANY_VALUE(JSON_VALUE(payload, '$.name_acronym')) AS code
  FROM `${project_id}.raw.raw_events`
  WHERE run_id = target_run AND event_type = 'drivers'
  GROUP BY driver_number
) AS d ON d.driver_number = m.driver_number
WHERE m.run_id = target_run
GROUP BY m.moment, m.lap, d.code, m.event_ts, m.detail, m.publish_time
ORDER BY m.event_ts;
"""
run_sql(MOMENTS)

## Step 4: Read the strategy calls

For a safety car, a rain start or a gap closing, the `strategist` service runs one BigQuery script: a snapshot of the field, then `AI.GENERATE` with Gemini. This is that script, with the lap 47 safety car of the latest replay filled in. Saved query **04 Strategist: the call for one moment**. Run it once the replay has passed lap 47.

In [ ]:
STRATEGIST = r"""
DECLARE target_run STRING DEFAULT (
  SELECT run_id FROM `${project_id}.serving.moments` ORDER BY publish_time DESC LIMIT 1);
DECLARE moment_name STRING DEFAULT 'SAFETY_CAR';
DECLARE moment_detail STRING DEFAULT 'SAFETY CAR DEPLOYED';
DECLARE moment_driver INT64 DEFAULT NULL;
DECLARE race_time TIMESTAMP DEFAULT TIMESTAMP '2025-03-16 05:41:06+00';
-- (the strategist service sets the five variables here)

WITH ev AS (
  SELECT event_type, event_ts, driver_number, payload
  FROM `${project_id}.raw.raw_events`
  WHERE run_id = target_run
    AND event_ts <= race_time
    AND event_type IN ('position', 'intervals', 'stints', 'laps', 'weather', 'drivers')
),

drivers AS (
  SELECT driver_number, ANY_VALUE(JSON_VALUE(payload, '$.name_acronym')) AS code
  FROM ev WHERE event_type = 'drivers' GROUP BY driver_number
),

positions AS (
  SELECT driver_number, ARRAY_AGG(SAFE.INT64(payload.position) ORDER BY event_ts DESC LIMIT 1)[OFFSET(0)] AS position
  FROM ev WHERE event_type = 'position' GROUP BY driver_number
),

gaps AS (
  SELECT
    driver_number,
    ARRAY_AGG(JSON_VALUE(payload, '$.gap_to_leader') ORDER BY event_ts DESC LIMIT 1)[OFFSET(0)] AS gap_to_leader,
    ARRAY_AGG(JSON_VALUE(payload, '$.interval') ORDER BY event_ts DESC LIMIT 1)[OFFSET(0)] AS interval_ahead
  FROM ev WHERE event_type = 'intervals' GROUP BY driver_number
),

stints AS (
  SELECT
    driver_number,
    ARRAY_AGG(STRUCT(
      JSON_VALUE(payload, '$.compound') AS compound,
      SAFE.INT64(payload.lap_start) AS lap_start,
      SAFE.INT64(payload.tyre_age_at_start) AS age_at_start
    ) ORDER BY event_ts DESC LIMIT 1)[OFFSET(0)] AS stint
  FROM ev WHERE event_type = 'stints' GROUP BY driver_number
),

laps AS (
  SELECT
    driver_number,
    MAX(SAFE.INT64(payload.lap_number)) AS lap,
    ARRAY_AGG(SAFE.FLOAT64(payload.lap_duration) IGNORE NULLS ORDER BY event_ts DESC LIMIT 3) AS last_laps
  FROM ev WHERE event_type = 'laps' GROUP BY driver_number
),

field AS (
  SELECT
    p.position,
    p.driver_number,
    d.code,
    g.gap_to_leader,
    g.interval_ahead,
    s.stint.compound AS compound,
    l.lap - s.stint.lap_start + s.stint.age_at_start AS tyre_age,
    l.last_laps,
    l.lap
  FROM positions AS p
  LEFT JOIN drivers AS d USING (driver_number)
  LEFT JOIN gaps AS g USING (driver_number)
  LEFT JOIN stints AS s USING (driver_number)
  LEFT JOIN laps AS l USING (driver_number)
),

context AS (
  SELECT
    MAX(lap) AS race_lap,
    STRING_AGG(
      FORMAT('P%d %s (#%d) gap %s interval %s %s tyres %s laps old, last laps %s',
        position, IFNULL(code, '?'), driver_number, IFNULL(gap_to_leader, '-'),
        IFNULL(interval_ahead, '-'), IFNULL(compound, '?'),
        IFNULL(CAST(tyre_age AS STRING), '?'), TO_JSON_STRING(last_laps)),
      '\n' ORDER BY position) AS field_text,
    (SELECT ARRAY_AGG(SAFE.INT64(payload.rainfall) ORDER BY event_ts DESC LIMIT 1)[OFFSET(0)]
      FROM ev WHERE event_type = 'weather') AS raining
  FROM field
)

SELECT
  race_lap,
  AI.GENERATE(
    CONCAT(
      '<role>You are the race strategist on an F1 pit wall.</role>\n',
      '<constraints>\n',
      '- Use only the cars, gaps and tyres in the context. Do NOT invent drivers or data.\n',
      '- At most 5 short lines, one per car that should act. Each line: driver code, BOX or STAY OUT, the tyre to fit if boxing, and why.\n',
      '- If the moment names a car, start with that car and its rivals around it.\n',
      '- If no car should change plan, say so in one line.\n',
      '</constraints>\n',
      '<context>\n',
      'Race: 57 laps. Current lap: ', CAST(race_lap AS STRING), '. Track: ',
      IF(raining = 1, 'raining', 'dry'), '.\n',
      'Field (position, driver, gap to leader, interval to car ahead, tyre compound, tyre age, last 3 lap times in seconds):\n',
      field_text, '\n',
      '</context>\n',
      '<task>Based on the information above, the moment is: ', moment_name,
      IFNULL(CONCAT(' for ', (SELECT code FROM drivers WHERE driver_number = moment_driver),
        ' (#', CAST(moment_driver AS STRING), ')'), ''),
      ' (', moment_detail, '). Give the strategy call.</task>'),
    connection_id => '${project_id}.us.vertex-ai-connection',
    -- gemini-3.8-flash: retirement date "September 2, 2027 or later", from
    -- https://docs.cloud.google.com/gemini-enterprise-agent-platform/models/gemini/3-8-flash
    -- (read on 2026-10-25). Migration tracked in issue #2.
    endpoint => 'https://aiplatform.googleapis.com/v1/projects/${project_id}/locations/global/publishers/google/models/gemini-3.8-flash',
    model_params => JSON '{"generationConfig": {"thinkingConfig": {"thinkingLevel": "LOW"}}}'
  ) AS call
FROM context;
"""
run_sql(STRATEGIST)

All the calls of the latest replay. Saved query **04b Calls: the strategist's answers**.

In [ ]:
CALLS = r"""
DECLARE target_run STRING DEFAULT (
  SELECT run_id FROM `${project_id}.raw.raw_events`
  WHERE event_type = 'drivers'
  ORDER BY publish_time DESC LIMIT 1);

SELECT
  c.moment,
  c.lap,
  d.code AS driver,
  c.event_ts AS race_time,
  c.status,
  ROUND(c.latency_ms / 1000, 1) AS seconds,
  c.call_text,
  c.error
FROM `${project_id}.serving.calls` AS c
LEFT JOIN (
  SELECT driver_number, ANY_VALUE(JSON_VALUE(payload, '$.name_acronym')) AS code
  FROM `${project_id}.raw.raw_events`
  WHERE run_id = target_run AND event_type = 'drivers'
  GROUP BY driver_number
) AS d ON d.driver_number = c.driver_number
WHERE c.run_id = target_run
ORDER BY c.event_ts;
"""
run_sql(CALLS)

## Step 5: Prove it

Compare the moments the continuous queries flagged with what the race contains, worked out from the raw events with a batch query. Run it after the replay ends. Every row should say `ok = True`. Saved query **05 Proof: flagged moments against the race**.

In [ ]:
PROOF = r"""
DECLARE target_run STRING DEFAULT (
  SELECT run_id FROM `${project_id}.raw.raw_events`
  WHERE event_type = 'drivers'
  ORDER BY publish_time DESC LIMIT 1);

WITH expected AS (
  SELECT 'SAFETY_CAR' AS moment, COUNT(*) AS n
  FROM `${project_id}.raw.raw_events`
  WHERE run_id = target_run AND event_type = 'race_control'
    AND JSON_VALUE(payload, '$.category') = 'SafetyCar'
    AND JSON_VALUE(payload, '$.message') LIKE '%DEPLOYED%'
  UNION ALL
  SELECT 'PIT_STOP', COUNT(*)
  FROM `${project_id}.raw.raw_events`
  WHERE run_id = target_run AND event_type = 'pit'
  UNION ALL
  SELECT 'RAIN_STARTED', COUNTIF(rainfall = 1 AND previous = 0)
  FROM (
    SELECT
      SAFE.INT64(payload.rainfall) AS rainfall,
      LAG(SAFE.INT64(payload.rainfall)) OVER (ORDER BY event_ts) AS previous
    FROM `${project_id}.raw.raw_events`
    WHERE run_id = target_run AND event_type = 'weather'
  )
),

flagged AS (
  -- A moment can be flagged twice after a continuous query restarts, so count
  -- each moment once. Only moments within the race count: the smoke test in
  -- scripts/verify.sh publishes a moment stamped with today's date.
  SELECT moment, COUNT(DISTINCT TO_JSON_STRING(STRUCT(driver_number, event_ts))) AS n
  FROM `${project_id}.serving.moments`
  WHERE run_id = target_run
    AND event_ts <= (
      SELECT MAX(event_ts) FROM `${project_id}.raw.raw_events` WHERE run_id = target_run)
  GROUP BY moment
)

SELECT
  target_run AS run_id,
  e.moment,
  e.n AS expected,
  IFNULL(f.n, 0) AS flagged,
  e.n = IFNULL(f.n, 0) AS ok
FROM expected AS e
LEFT JOIN flagged AS f USING (moment)
ORDER BY e.moment;
"""
run_sql(PROOF)

## Stop

The reservation bills while a continuous query runs. Cancel both when the replay is over. Saved query **99 Stop: cancel the continuous queries** does the same.

In [ ]:
for job_id in JOB_IDS:
    session.post(
        f"https://bigquery.googleapis.com/bigquery/v2/projects/{PROJECT_ID}/jobs/{job_id}/cancel",
        params={"location": BQ_LOCATION},
    ).raise_for_status()
    print("Cancelled", job_id)